# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
from typing import List

import chromadb
from chromadb.utils import embedding_functions
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [4]:
# Environment variables are loaded in the setup cell above.

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [5]:
# Load the vector DB built in Part 1 (same path, collection name and embedding function)
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)
print(f"Loaded collection 'udaplay' with {collection.count()} documents")

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

Loaded collection 'udaplay' with 15 documents


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [6]:
@tool
def retrieve_game(query: str) -> List[dict]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    try:
        results = collection.query(query_texts=[query], n_results=3)
    except Exception as e:
        return [{"error": f"Vector DB query failed: {e}"}]

    docs = []
    for metadata, distance in zip(results["metadatas"][0], results["distances"][0]):
        docs.append({
            "Platform": metadata.get("Platform"),
            "Name": metadata.get("Name"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description"),
            "distance": round(distance, 4),
        })
    return docs

retrieve_game("When was Pokémon Gold and Silver released?")

[{'Platform': 'Game Boy Color',
  'Name': 'Pokémon Gold and Silver',
  'YearOfRelease': 1999,
  'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.',
  'distance': 0.1128},
 {'Platform': 'Game Boy Advance',
  'Name': 'Pokémon Ruby and Sapphire',
  'YearOfRelease': 2002,
  'Description': 'Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.',
  'distance': 0.1467},
 {'Platform': 'Nintendo 64',
  'Name': 'Super Mario 64',
  'YearOfRelease': 1996,
  'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.",
  'distance': 0.2207}]

#### Evaluate Retrieval Tool

In [7]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Description about the evaluation result")


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[dict]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    judge = LLM(model="gpt-4o-mini", temperature=0.0)
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n\n"
        f"Retrieved documents:\n{json.dumps(retrieved_docs, indent=2, ensure_ascii=False)}"
    )
    try:
        response = judge.invoke(prompt, response_format=EvaluationReport)
        report = EvaluationReport.model_validate_json(response.content)
    except Exception as e:
        return {"useful": False, "description": f"Evaluation failed: {e}"}
    return report.model_dump()

evaluate_retrieval(
    "When was Pokémon Gold and Silver released?",
    retrieve_game("When was Pokémon Gold and Silver released?"),
)

{'useful': True,
 'description': 'The retrieved documents contain relevant information regarding the release date of Pokémon Gold and Silver. Specifically, the first document explicitly states that Pokémon Gold and Silver were released in 1999 for the Game Boy Color. This directly answers the query about the release date of the game. The other documents, while related to different Pokémon games, do not provide any conflicting information and are not necessary for answering the question. Therefore, the documents are sufficient to respond to the query.'}

#### Game Web Search Tool

In [8]:
@tool
def game_web_search(question: str) -> dict:
    """
    Web search: searches the web with Tavily when the vector DB results aren't good enough
    args:
    - question: a question about game industry.

    Returns the search results, including each page's URL, so the answer can cite its source.
    """
    try:
        response = tavily_client.search(
            query=question,
            search_depth="advanced",
            max_results=5,
            include_answer=True,
        )
    except Exception as e:
        return {"error": f"Web search failed: {e}"}

    return {
        "answer": response.get("answer"),
        "results": [
            {"title": r.get("title"), "url": r.get("url"), "content": r.get("content")}
            for r in response.get("results", [])
        ],
    }

### Agent

In [9]:
instructions = """
You are UdaPlay, an AI research agent specialized in the video game industry.

Follow this workflow for every question:
1. ALWAYS call `retrieve_game` first to search the internal game database.
2. Call `evaluate_retrieval` with the user's question and the retrieved documents.
3. If the evaluation says the documents are useful, answer from them.
   Otherwise, call `game_web_search` to find the information on the web.
4. Never make up facts. If neither source has the answer, say so.

Answer format:
- Give a clear, concise answer to the question.
- End with a "Source:" line that cites exactly what you used:
  - internal database: "Internal database: <Name> (<Platform>, <YearOfRelease>)"
  - web: the URL(s) of the web pages you used
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)

In [10]:
def ask(question: str, session_id: str = "udaplay"):
    """Run the agent and print this turn's reasoning, tool usage and final answer."""
    run = agent.invoke(question, session_id=session_id)
    messages = run.get_final_state()["messages"]

    # Only look at this turn: everything after the latest user message
    last_user = max(i for i, m in enumerate(messages) if isinstance(m, UserMessage))
    turn = messages[last_user + 1:]

    print(f"Q: {question}")
    tool_calls = [tc for m in turn if isinstance(m, AIMessage) and m.tool_calls for tc in m.tool_calls]
    if not tool_calls:
        print("Tools used: none (answered from conversation history)")
    for tc in tool_calls:
        print(f"Tool call: {tc.function.name}({tc.function.arguments[:120]})")

    for m in turn:
        if isinstance(m, ToolMessage) and m.name == "evaluate_retrieval":
            print(f"Reasoning (evaluate_retrieval): {json.loads(m.content)}")

    print(f"Final answer: {messages[-1].content}\n")
    print("-" * 80)
    return run

questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]
for q in questions:
    ask(q, session_id=q)  # separate session per question

# Conversation state: a follow-up in the same session refers back to "it"
ask("And what platform was it released on?", session_id=questions[1])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


...

[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q: Was Mortal Kombat X released for PlayStation 5?
Tool call: retrieve_game({"query":"Mortal Kombat X PlayStation 5"})
Tool call: evaluate_retrieval({"question":"Was Mortal Kombat X released for PlayStation 5?","retrieved_docs":["{'Platform': 'PlayStation 5', 'Name': \)
Tool call: game_web_search({"question":"Was Mortal Kombat X released for PlayStation 5?"})
Reasoning (evaluate_retrieval): {'useful': False, 'description': "The retrieved documents do not contain any information regarding the release of Mortal Kombat X for PlayStation 5. Instead, they focus on other games, specifically Marvel's Spider-Man 2, Marvel's Spider-Man, and Halo Infinite, none of which are related to Mortal Kombat X. Therefore, these documents are not useful for answering the question."}
Final answer: Mortal Kombat X was not released natively for PlayStation 5. It was originally released for PlayStation 4 on April 14, 2015.

[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q: And what platform was it released on?
Tools used: none (answered from conversation history)
Final answer: **Super Mario 64** was released on the **Nintendo 64** platform.

Source: Internal database: Super Mario 64 (Nintendo 64, 1996)

--------------------------------------------------------------------------------


Run('8e286560-e839-47cd-81b0-d93fa58a7b25')

### Report: agent performance on the example queries

Every query starts with **internal knowledge** (`retrieve_game`), then `evaluate_retrieval` (an LLM judge)
decides whether the retrieved documents are enough. Only when they aren't does the agent fall back to
`game_web_search` (Tavily). The tool calls and the judge's reasoning are printed above for each query.

| # | Query | Tools used | Judge's verdict | Final answer | Citation |
|---|---|---|---|---|---|
| 1 | When was Pokémon Gold and Silver released? | `retrieve_game` → `evaluate_retrieval` | useful | 1999, on Game Boy Color | Internal database: Pokémon Gold and Silver (Game Boy Color, 1999) |
| 2 | Which one was the first 3D platformer Mario game? | `retrieve_game` → `evaluate_retrieval` | useful | Super Mario 64 (Nintendo 64, 1996) | Internal database: Super Mario 64 (Nintendo 64, 1996) |
| 3 | Was Mortal Kombat X released for PlayStation 5? | `retrieve_game` → `evaluate_retrieval` → `game_web_search` | **not useful** (DB only had unrelated PS5 games) | No: released for PS4 (April 14, 2015); playable on PS5 via backward compatibility and the PlayStation Plus Collection | https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V |
| 4 | *(follow-up to #2)* And what platform was it released on? | none | n/a | Nintendo 64 | Internal database: Super Mario 64 (Nintendo 64, 1996) |

**Observations**
- **Internal knowledge first:** questions 1 and 2 were answered from the vector DB without any web call.
- **Web fallback works:** for question 3 the judge correctly rejected the retrieved documents (none were about
  Mortal Kombat X), so the agent searched the web and cited the page it used.
- **Conversation state:** question 4 says "it" without naming the game. The agent resolved it to Super Mario 64
  from the earlier turn in the same session and answered without calling any tool.
- **Limitation:** the internal DB has only 15 games, so anything outside it depends on web search quality;
  LLM wording varies slightly between runs, while the facts and tool paths stay the same.

### (Optional) Advanced — Stand Out

#### Long-term memory

`lib.memory.LongTermMemory` is in-memory unless it gets a persistent ChromaDB client, so we give it
`persist_path="memory_db"`. Facts the agent learns from the web are stored there and survive
kernel restarts, so the next time the same question comes up the agent can answer without searching the web again.

In [11]:
from typing import Literal, Optional, TypedDict

from lib.memory import LongTermMemory, MemoryFragment
from lib.state_machine import StateMachine, Step, EntryPoint, Termination
from lib.vector_db import VectorStoreManager

MEMORY_OWNER = "udaplay_user"
MEMORY_NAMESPACE = "game_facts"

memory_manager = VectorStoreManager(os.getenv("OPENAI_API_KEY"), persist_path="memory_db")
long_term_memory = LongTermMemory(memory_manager)

# Start the demo from an empty memory so it's reproducible; comment this out to keep
# what earlier sessions learned. We delete the documents rather than using reset=True,
# which drops the whole collection and breaks any other kernel that has memory_db open.
long_term_memory.vector_store._collection.delete(where={"owner": MEMORY_OWNER})

{'deleted': 1}

#### Agent as a state machine

Instead of letting the LLM decide which tool to call, each tool is a pre-defined node and the
transitions encode the workflow:

```
entry → recall_memory → retrieve_game → evaluate_retrieval ─┬─ useful ─────────────→ generate_answer → store_memory → end
                                                            └─ not useful → game_web_search ─┘
```

`generate_answer` returns a structured `AgentAnswer` (Pydantic), and `store_memory` saves
web-sourced answers to long-term memory.

In [12]:
class AgentAnswer(BaseModel):
    answer: str = Field(description="Clear, concise answer to the user's question")
    source: Literal["internal_database", "long_term_memory", "web_search", "unknown"] = Field(
        description="Where the answer came from"
    )
    citations: List[str] = Field(default_factory=list, description="URLs or internal database records used")


class UdaPlayState(TypedDict):
    question: str
    memories: List[str]              # facts recalled from long-term memory
    retrieved_docs: List[dict]       # games found in the vector DB
    evaluation: Optional[dict]       # EvaluationReport as dict
    web_results: Optional[dict]      # Tavily results, only if the DB wasn't enough
    final_answer: Optional[AgentAnswer]
    path: List[str]                  # nodes visited, for transparency


def recall_memory_step(state: UdaPlayState) -> dict:
    result = long_term_memory.search(
        state["question"], owner=MEMORY_OWNER, namespace=MEMORY_NAMESPACE, limit=3
    )
    return {
        "memories": [f.content for f in result.fragments],
        "path": state["path"] + ["recall_memory"],
    }


def retrieve_game_step(state: UdaPlayState) -> dict:
    return {
        "retrieved_docs": retrieve_game(state["question"]),
        "path": state["path"] + ["retrieve_game"],
    }


def evaluate_retrieval_step(state: UdaPlayState) -> dict:
    # Recalled memories count as retrieved knowledge too
    docs = state["retrieved_docs"] + [{"memory": m} for m in state["memories"]]
    return {
        "evaluation": evaluate_retrieval(state["question"], docs),
        "path": state["path"] + ["evaluate_retrieval"],
    }


def game_web_search_step(state: UdaPlayState) -> dict:
    return {
        "web_results": game_web_search(state["question"]),
        "path": state["path"] + ["game_web_search"],
    }


def generate_answer_step(state: UdaPlayState) -> dict:
    context = {
        "internal_database": state["retrieved_docs"],
        "long_term_memory": state["memories"],
        "web_search": state["web_results"],
    }
    llm = LLM(model="gpt-4o-mini", temperature=0.0)
    messages = [
        SystemMessage(content=(
            "You are UdaPlay, an AI research agent for the video game industry. "
            "Answer the question using ONLY the context provided. Prefer web_search when present, "
            "since it is fresher. Ignore context that is unrelated to the question. "
            "If the context doesn't contain the answer, say so and set source to 'unknown'. "
            "In citations, put the URLs you relied on, and for internal_database records "
            "'Internal database: <Name> (<Platform>, <YearOfRelease>)'."
        )),
        UserMessage(content=(
            f"Question: {state['question']}\n\n"
            f"Context:\n{json.dumps(context, indent=2, ensure_ascii=False)}"
        )),
    ]
    response = llm.invoke(messages, response_format=AgentAnswer)
    return {
        "final_answer": AgentAnswer.model_validate_json(response.content),
        "path": state["path"] + ["generate_answer"],
    }


def store_memory_step(state: UdaPlayState) -> dict:
    answer = state["final_answer"]
    # Only web-sourced answers are new knowledge worth remembering
    if answer.source == "web_search":
        long_term_memory.register(MemoryFragment(
            content=f"Q: {state['question']}\nA: {answer.answer} (sources: {', '.join(answer.citations)})",
            owner=MEMORY_OWNER,
            namespace=MEMORY_NAMESPACE,
        ))
    return {"path": state["path"] + ["store_memory"]}


def build_udaplay_machine() -> StateMachine[UdaPlayState]:
    machine = StateMachine[UdaPlayState](UdaPlayState)

    entry = EntryPoint[UdaPlayState]()
    recall = Step[UdaPlayState]("recall_memory", recall_memory_step)
    retrieve = Step[UdaPlayState]("retrieve_game", retrieve_game_step)
    evaluate = Step[UdaPlayState]("evaluate_retrieval", evaluate_retrieval_step)
    web = Step[UdaPlayState]("game_web_search", game_web_search_step)
    answer = Step[UdaPlayState]("generate_answer", generate_answer_step)
    store = Step[UdaPlayState]("store_memory", store_memory_step)
    end = Termination[UdaPlayState]()

    machine.add_steps([entry, recall, retrieve, evaluate, web, answer, store, end])

    machine.connect(entry, recall)
    machine.connect(recall, retrieve)
    machine.connect(retrieve, evaluate)
    machine.connect(
        evaluate, [answer, web],
        lambda state: answer if state["evaluation"].get("useful") else web,
    )
    machine.connect(web, answer)
    machine.connect(answer, store)
    machine.connect(store, end)
    return machine


udaplay_machine = build_udaplay_machine()


def ask_stateful(question: str) -> AgentAnswer:
    run = udaplay_machine.run({
        "question": question,
        "memories": [],
        "retrieved_docs": [],
        "evaluation": None,
        "web_results": None,
        "final_answer": None,
        "path": [],
    })
    state = run.get_final_state()
    answer = state["final_answer"]
    print(f"\nQ: {question}")
    print(f"Path: {' → '.join(state['path'])}")
    print(f"Reasoning (evaluate_retrieval): useful={state['evaluation'].get('useful')} - {state['evaluation'].get('description')}")
    print(f"A: {answer.answer}")
    print(f"Source: {answer.source} {answer.citations}\n")
    return answer

In [13]:
for q in questions:
    ask_stateful(q)

[StateMachine] Starting: __entry__


[StateMachine] Executing step: recall_memory


[StateMachine] Executing step: retrieve_game


[StateMachine] Executing step: evaluate_retrieval


[StateMachine] Executing step: generate_answer
[StateMachine] Executing step: store_memory
[StateMachine] Terminating: __termination__

Q: When was Pokémon Gold and Silver released?
Path: recall_memory → retrieve_game → evaluate_retrieval → generate_answer → store_memory
Reasoning (evaluate_retrieval): useful=True - The retrieved documents contain relevant information to answer the query about the release date of Pokémon Gold and Silver. Specifically, the first document explicitly states that Pokémon Gold and Silver were released in 1999 for the Game Boy Color. This directly addresses the question asked. The other documents, while providing information about different games, do not pertain to the query and can be disregarded. Therefore, the information provided is sufficient to respond to the question.
A: Pokémon Gold and Silver were released in 1999.
Source: internal_database ['Internal database: Pokémon Gold and Silver (Game Boy Color, 1999)']

[StateMachine] Starting: __entry__


...

[StateMachine] Executing step: retrieve_game


[StateMachine] Executing step: evaluate_retrieval


[StateMachine] Executing step: game_web_search


[StateMachine] Executing step: generate_answer


[StateMachine] Executing step: store_memory
[StateMachine] Terminating: __termination__

Q: Was Mortal Kombat X released for PlayStation 5?
Path: recall_memory → retrieve_game → evaluate_retrieval → game_web_search → generate_answer → store_memory
Reasoning (evaluate_retrieval): useful=False - The retrieved documents do not contain any information regarding the release of Mortal Kombat X for PlayStation 5. Instead, they focus on other games, such as Marvel's Spider-Man 2 and Halo Infinite, which are unrelated to the query. Therefore, the documents are insufficient to answer whether Mortal Kombat X was released for PlayStation 5.
A: Mortal Kombat X was not released natively for PlayStation 5, but it is playable on the PS5 through backward compatibility with the PS4 version. It is also included in the PlayStation Plus Collection for PS5 subscribers.
Source: web_search ['https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V', 'htt

Ask the web-sourced question again. Long-term memory now contains the answer, so
`evaluate_retrieval` should judge it sufficient and the agent should skip `game_web_search`.

In [14]:
ask_stateful("Was Mortal Kombat X released for PlayStation 5?")

[StateMachine] Starting: __entry__


[StateMachine] Executing step: recall_memory


[StateMachine] Executing step: retrieve_game


[StateMachine] Executing step: evaluate_retrieval


[StateMachine] Executing step: generate_answer
[StateMachine] Executing step: store_memory
[StateMachine] Terminating: __termination__

Q: Was Mortal Kombat X released for PlayStation 5?
Path: recall_memory → retrieve_game → evaluate_retrieval → generate_answer → store_memory
Reasoning (evaluate_retrieval): useful=True - The retrieved documents contain relevant information that directly answers the query about whether Mortal Kombat X was released for PlayStation 5. The memory entry explicitly states that Mortal Kombat X was not released natively for PlayStation 5 but is playable on the PS5 through backward compatibility with the PS4 version. Additionally, it mentions that the game is included in the PlayStation Plus Collection for PS5 subscribers. This information is sufficient to conclude that while Mortal Kombat X was not released specifically for the PS5, it is accessible on the platform, which addresses the user's question adequately.
A: Mortal Kombat X was not released natively fo

AgentAnswer(answer='Mortal Kombat X was not released natively for PlayStation 5, but it is playable on the PS5 through backward compatibility with the PS4 version. It is also included in the PlayStation Plus Collection for PS5 subscribers.', source='long_term_memory', citations=['https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V', 'https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved'])